# Chapter 4 &mdash; The Pumping Lemma in Predicate Logic

**Concept 22 of the Chapter 4 decomposition:** *The Pumping Lemma in Predicate Logic, and a More General Version*

The lemma with explicit quantifiers, and the more general version with an arbitrary head and tail.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Pumping-Lemma-Predicate-Logic/Concept-Pumping-Lemma-Predicate-Logic.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$Reg(L) \Rightarrow \exists N : \forall w \in L : [\,|w|\ge N \Rightarrow \exists x,y,z :
w=xyz \wedge |xy|\le N \wedge y\ne\varepsilon \wedge \forall i\ge0 : xy^iz\in L\,]$$

Reliable use needs the symbolic form &mdash; negation is then mechanical.

**The more general lemma.** Concept 13 focused on the *first* pump only for crispness.
Drop that and you may split **any** length-$N$ window: $w = hmt$ with arbitrary head
$h$, middle $m$ of length $N$, arbitrary tail $t$, and split $m$ into $xyz$ every way.

That version settles $L_{if}$ directly &mdash; put the window inside the $b$s.

## 2. Definitions

### The general lemma, as a checker

In [ ]:
def in_Lif(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    if rest[j:] != 'c'*k or s != 'a'*i + 'b'*j + 'c'*k: return False
    return (j == k) if i == 3 else True

def general_splits(w, N):
    """w = h m t with |m| = N, then m split into x y z with y non-empty."""
    out = []
    for hs in range(len(w) - N + 1):
        h, m, t = w[:hs], w[hs:hs+N], w[hs+N:]
        for p in range(N+1):
            for q in range(p+1, N+1):
                out.append((h + m[:p], m[p:q], m[q:] + t))
    return out

### The simple version, for comparison

In [ ]:
def simple_splits(w, N):
    return [(w[:p], w[p:q], w[q:])
            for p in range(N+1) for q in range(p+1, min(N,len(w))+1)]

## 3. Tests

The simple lemma leaves survivors on $L_{if}$ &mdash; as Concept 20 showed.

In [ ]:
N = 4; w = 'aaa' + 'b'*N + 'c'*N
surv = [(x,y,z) for (x,y,z) in simple_splits(w,N)
        if all(in_Lif(x + y*i + z) for i in range(4))]
print("simple lemma, surviving splits :", len(surv))
assert surv

The **general** lemma lets you place the window in the $b$s &mdash; and then it breaks.

In [ ]:
windows = general_splits(w, N)
bs = [(x,y,z) for (x,y,z) in windows if set(y) <= {'b'}]
broken = [(x,y,z) for (x,y,z) in bs if any(not in_Lif(x+y*i+z) for i in range(4))]
print("splits with y inside the b-block :", len(bs))
print("of those, broken by pumping      :", len(broken))
assert bs and len(broken) == len(bs)
print("\nChoosing the window is the extra freedom the general lemma buys.")

The JFLAP line-up: sort the regular from the non-regular by eye.

In [ ]:
lineup = [("L1  0^i 1^i",            "NOT regular -- needs unbounded counting"),
          ("L4  a^n b^k c^(n+k)",    "NOT regular -- ditto"),
          ("L6  a^n, n even",        "REGULAR -- parity is a bounded residue"),
          ("L10 b^5 w, (2#a+5#b)%3", "REGULAR -- a residue mod 3"),
          ("L13 if i=3 then j=k",    "NOT regular -- but resists the simple lemma")]
for name, verdict in lineup:
    print("%-26s %s" % (name, verdict))
print("\nDiagnostic: does the condition need a COUNT, or only a bounded RESIDUE?")

## 4. Exercises


1. Negate the predicate-logic form mechanically and check it against the English recipe.
2. State the general lemma yourself, with $h$, $m$ and $t$ named.
3. Work the JFLAP line-up: build a DFA for each regular one, prove each other one.

In [ ]:
# Your work for the exercises above.